<a href="https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [ ]:
!pwd
!ls

/content
sample_data


In [ ]:
!git clone https://github.com/Fizzah-Amir14/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (182/182), done.
remote: Total 230 (delta 113), reused 101 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (230/230), 1.95 MiB | 13.48 MiB/s, done.
Resolving deltas: 100% (113/113), done.
/content/flyrank-ml-internship


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("shape:", df.shape)
print(df.columns.tolist())

shape: (30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [ ]:
# --- Adaptive column resolver ---
# Finds the first column whose name contains ALL given keywords (case-insensitive).
def find_col(df, *keywords, required=True):
    for c in df.columns:
        lc = c.lower()
        if all(k in lc for k in keywords):
            return c
    if required:
        raise KeyError(f"No column found matching {keywords}. Available: {list(df.columns)}")
    return None

# Core signals for Lane 4 (CTR / engagement decay)
col_ctr        = find_col(df, "ctr")
col_position   = find_col(df, "posit")
col_staleness  = find_col(df, "day", "updat", required=False) or find_col(df, "day", "sinc", required=False) or find_col(df, "staleness", required=False)
col_volume     = find_col(df, "query", "count", required=False) or find_col(df, "visible", "query", required=False)
col_trend_dir  = find_col(df, "trend_direction", required=False) or find_col(df, "trend", "direct", required=False)

print("CTR column:       ", col_ctr)
print("Position column:  ", col_position)
print("Staleness column: ", col_staleness)
print("Volume column:    ", col_volume)
print("Trend/label col:  ", col_trend_dir)

# Label — DO NOT use trend_direction / trend_pct as a feature, only to build the label for checking our rule against reality
df["is_declining_label"] = (df[col_trend_dir].astype(str).str.lower() == "down").astype(int)

CTR column:        ctr
Position column:   avg_position
Staleness column:  days_since_last_update
Volume column:     None
Trend/label col:   trend_direction


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# Bucket by position range, look at mean CTR + n per bucket
position_bins = [0, 3, 10, 20, 50, np.inf]
position_labels = ["1-3", "4-10", "11-20", "21-50", "50+"]
df["position_bucket"] = pd.cut(df[col_position], bins=position_bins, labels=position_labels)

signal_a = df.groupby("position_bucket", observed=True).agg(
    mean_ctr=(col_ctr, "mean"),
    n=(col_ctr, "size"),
).reset_index()
print(signal_a)

# Verdict: expected pattern is CTR decreasing as position worsens (better rank -> higher CTR)
is_monotonic_decreasing = signal_a["mean_ctr"].is_monotonic_decreasing
print("\nVerdict:", "CONFIRMED" if is_monotonic_decreasing else "MIXED",
      "— CTR", "decreases" if is_monotonic_decreasing else "does not cleanly decrease",
      "as position worsens, across n =", signal_a["n"].sum(), "pages.")

  position_bucket  mean_ctr      n
0             1-3  2.714303   1141
1            4-10  0.651045  11842
2           11-20  0.323443   7273
3           21-50  0.222345   7225
4             50+  0.150784   1314

Verdict: CONFIRMED — CTR decreases as position worsens, across n = 28795 pages.


In [ ]:
if col_staleness is not None:
    stale_bins = [0, 30, 90, 180, 365, np.inf]
    stale_labels = ["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]
    df["staleness_bucket"] = pd.cut(df[col_staleness], bins=stale_bins, labels=stale_labels)

    signal_b = df.groupby("staleness_bucket", observed=True).agg(
        decline_rate=("is_declining_label", "mean"),
        n=("is_declining_label", "size"),
    ).reset_index()
    print(signal_b)

    is_increasing = signal_b["decline_rate"].is_monotonic_increasing
    print("\nVerdict:", "CONFIRMED" if is_increasing else "MIXED",
          "— decline rate", "rises" if is_increasing else "does not cleanly rise",
          "with staleness, across n =", signal_b["n"].sum(), "pages.")
else:
    print("No staleness/last-updated column found in this dataset — swap in the closest")
    print("equivalent from your data dictionary and re-run this cell.")
    print("Available columns:", df.columns.tolist())

  staleness_bucket  decline_rate      n
0            0-30d      0.511377  20480
1           31-90d      0.588571    175
2          91-180d      0.611057   9171
3         181-365d      0.467456    169
4            365d+      0.600000      5

Verdict: MIXED — decline rate does not cleanly rise with staleness, across n = 30000 pages.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Volume signal — a page with almost no impressions can show ctr=0 without a real engagement
# problem; it's a data-insufficiency case, not an action candidate. Gate on volume before scoring.
col_volume = "impressions_last_30d"  # confirmed present in your columns

expected_ctr = df.groupby("position_bucket", observed=True)[col_ctr].transform("mean")
df["ctr_gap"] = expected_ctr - df[col_ctr]

MIN_IMPRESSIONS = 30  # below this, we don't trust the CTR reading enough to act on it

def score_row(row):
    score = 0.0
    reason = "LOW_SIGNAL"

    if row[col_volume] < MIN_IMPRESSIONS:
        reason = "INSUFFICIENT_VOLUME"
        return pd.Series({"action_score": 0.0, "reason_code": reason})

    if row["ctr_gap"] > 0:
        score += row["ctr_gap"] * 100
        reason = "CTR_BELOW_EXPECTED_FOR_POSITION"
    if col_staleness is not None and pd.notna(row.get(col_staleness, np.nan)):
        if row[col_staleness] > 180 and row[col_position] <= 20:
            score += 10
            if reason == "LOW_SIGNAL":
                reason = "STALE_HIGH_POSITION"
    return pd.Series({"action_score": score, "reason_code": reason})

df[["action_score", "reason_code"]] = df.apply(score_row, axis=1)

# Break ties within the same score using actual impression volume, so higher-traffic pages surface first
df["action_score_tiebreak"] = df["action_score"] + (df[col_volume] / df[col_volume].max()) * 0.01

def action_label(score):
    if score >= 15:
        return "Refresh Now"
    elif score >= 5:
        return "Monitor"
    return "No Action"

df["action_label"] = df["action_score"].apply(action_label)

queue = df.sort_values("action_score_tiebreak", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote", len(queue), "ranked rows to work/outputs/baseline_action_score.csv")
queue[[col_position, col_ctr, col_volume, "action_score", "reason_code", "action_label"]].head(10)

Wrote 30000 ranked rows to work/outputs/baseline_action_score.csv


,avg_position,ctr,impressions_last_30d,action_score,reason_code,action_label
0,2.4,0.0,1444,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
1,1.5,0.0,303,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
2,2.7,0.0,245,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
3,1.6,0.0,232,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
4,1.8,0.0,212,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
5,1.8,0.0,197,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
6,2.6,0.0,184,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
7,2.3,0.0,172,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
8,2.8,0.0,139,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
9,2.7,0.0,136,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:

top20 = queue.head(20).copy()
top20["confidence_note"] = np.where(top20["action_score"] >= 20, "high", "moderate")
top20["what_would_make_it_wrong"] = ""  # <-- fill this in per row, one honest line each

top20[[col_position, col_ctr, "action_score", "reason_code", "action_label",
       "confidence_note", "what_would_make_it_wrong"]]

,avg_position,ctr,action_score,reason_code,action_label,confidence_note,what_would_make_it_wrong
0,2.4,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
1,1.5,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
2,2.7,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
3,1.6,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
4,1.8,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
5,1.8,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
6,2.6,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
7,2.3,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
8,2.8,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
9,2.7,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Leakage check: trend_direction / trend_pct must NEVER be inputs to the score
feature_inputs_used = [col_ctr, col_position, col_staleness]
leak_terms = ["trend_direction", "trend_pct"]
leaked = [c for c in feature_inputs_used if c and any(t in c.lower() for t in leak_terms)]
assert not leaked, f"Leakage detected: {leaked}"
print("Leak check passed — no future-window or label-derived inputs used in scoring.")

# Look for weak/wrong-looking picks in the top 20 (e.g. very low n context, edge-case positions)
weak_candidates = top20[top20["action_score"] < top20["action_score"].median()]
print(f"\n{len(weak_candidates)} of the top 20 sit below the median score within that group —")
print("review these first for false positives:")
weak_candidates[[col_position, col_ctr, "action_score", "reason_code"]]

Leak check passed — no future-window or label-derived inputs used in scoring.

0 of the top 20 sit below the median score within that group —
review these first for false positives:


,avg_position,ctr,action_score,reason_code


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.